# Regional VMD EOG removal — regional-context-match

Isolate raw versus VMD frontal artifact context for central transfer.

## Scientific question, inputs and contracts

Attach corpus/2, regional-full/1, diagnostic-full/1 and verified central-coverage-full/1, with explicit manifest identities. Reuse every old scored output. Fit exactly one additional central raw-artifact-context arm on the identical four OOF calibration chunks and ridge policy. Frontal and posterior scored outputs remain fixed VMD/direct outputs for both context arms. Raw scoring context comes from the archived calibrated raw frontal artifact estimate, not a paired target or stored mixture amplitude. No new VMD decompositions, AutoVMD, recipe search, neural models or confirmation access. Save whole/cohort/condition/region paper metrics, preservation failures, fit availability and recipient/donor-aware project uncertainty. Identical clean outputs do not qualify native preservation.

## Fresh source and environment
Resolve latest published code once and pin it. New temporary clones avoid stale Python imports; original inputs remain immutable.

In [ ]:
import os, sys, json, tempfile, subprocess
from pathlib import Path
IN_KAGGLE = Path('/kaggle/input').is_dir()
if not IN_KAGGLE:
    raise RuntimeError('This numerical campaign runs on Kaggle only')
TEMP_ROOT = Path(tempfile.mkdtemp(prefix='vmd-eog-'))
CHECKOUT = TEMP_ROOT / 'repository'
REPOSITORY_URL = 'https://github.com/saltypal/Lateral-eye-Artifact-removal.git'
BRANCH = 'research/vmd-eog-removal-notebooks'
RUN_SPEC = {'stage': 'regional-context-match', 'run_id': 'context-match-manual', 'campaign_id': 'vmd-context-match-20261013', 'git_sha': '', 'profile': 'pilot', 'config_path': 'configs/context_match.json'}
OUT = Path('/kaggle/working/results') / RUN_SPEC['campaign_id'] / RUN_SPEC['run_id']
OUT.mkdir(parents=True,exist_ok=False)
(OUT/'run_spec.json').write_text(json.dumps(RUN_SPEC,indent=2))
(OUT/'experiment.json').write_text(json.dumps(RUN_SPEC.get('experiment',{}),indent=2))
(OUT/'execution_state.json').write_text(json.dumps({'stage':RUN_SPEC['stage'],'status':'bootstrap_running'}))
print('BOOTSTRAP_START', RUN_SPEC['run_id'], RUN_SPEC['git_sha'], flush=True)
def bootstrap_call(command, phase):
    print('BOOTSTRAP_PHASE', RUN_SPEC['run_id'], phase, flush=True)
    result = subprocess.run(command)
    if result.returncode:
        (OUT/'execution_state.json').write_text(json.dumps({'stage':RUN_SPEC['stage'],
            'status':'failed','failure_layer':'bootstrap','phase':phase,'exit_code':result.returncode}))
        raise RuntimeError('Bootstrap failed during '+phase+'; retained run identity and status in '+str(OUT))
# Manual launches resolve latest once; CLI launches already have its exact SHA.
if not RUN_SPEC['git_sha']:
    line = subprocess.check_output(['git','ls-remote',REPOSITORY_URL,'refs/heads/'+BRANCH], text=True).strip()
    RUN_SPEC['git_sha'] = line.split()[0]
bootstrap_call(['git','clone','--quiet','--branch',BRANCH,'--single-branch',REPOSITORY_URL,str(CHECKOUT)], 'clone')
bootstrap_call(['git','-C',str(CHECKOUT),'checkout','--quiet',RUN_SPEC['git_sha']], 'checkout')
ACTUAL_SHA = subprocess.check_output(['git','-C',str(CHECKOUT),'rev-parse','HEAD'],text=True).strip()
if ACTUAL_SHA != RUN_SPEC['git_sha']:
    raise RuntimeError('Source SHA mismatch')
ENV_ROOT = TEMP_ROOT / 'environment'
bootstrap_call([sys.executable,'-m','venv','--without-pip','--system-site-packages',str(ENV_ROOT)], 'environment')
ENV_PY = ENV_ROOT / 'bin/python'
bootstrap_call([sys.executable,'-m','pip','--python',str(ENV_PY),'install','--quiet','--progress-bar','off','-r',str(CHECKOUT/'requirements.txt')], 'dependencies')
PROCESS_ENV = os.environ.copy()
PROCESS_ENV['PYTHONPATH'] = str(CHECKOUT/'src')
PROCESS_ENV['PYTHONUNBUFFERED'] = '1'
PROCESS_ENV['OMP_NUM_THREADS'] = '2'
(OUT/'run_spec.json').write_text(json.dumps(RUN_SPEC,indent=2))
(OUT/'experiment.json').write_text(json.dumps(RUN_SPEC.get('experiment',{}),indent=2))
print('Exact Git SHA:', ACTUAL_SHA, 'Stage:', RUN_SPEC['stage'], 'Output:',OUT,flush=True)


## Inspect the pinned algorithm contract
The source modules remain inspectable in this checkout. Mode vectors, references and paired targets have distinct roles; the accompanying explanation defines their shapes and inference boundaries.

In [ ]:
CONFIGURATION = json.loads((CHECKOUT/RUN_SPEC.get('config_path','configs/campaign.json')).read_text())
print('Declared signal contract:', {key: CONFIGURATION[key] for key in ('fs','window','hop','seed')})
print('Preservation limits:', CONFIGURATION['preservation'])
if CONFIGURATION.get('schema_version') == 4:
    if RUN_SPEC['stage'] == 'regional-context-match':
        print('Modules: regional_context_match.py, calibration_frontal_context.py, coverage_uncertainty.py')
        print((CHECKOUT/'docs/MATCHED_CONTEXT_PLAN.md').read_text())
    elif RUN_SPEC['stage'] == 'regional-coverage':
        print('Modules: regional_coverage.py, nonfrontal_ridge.py, calibration_frontal_context.py')
        print((CHECKOUT/'docs/CENTRAL_COVERAGE_PLAN.md').read_text())
    elif RUN_SPEC['stage'] == 'regional-diagnostic':
        print('Modules: regional_diagnostic.py, paper_metrics.py, artifacts.py')
        print((CHECKOUT/'docs/REGIONAL_NEXT_STEPS_20261013.md').read_text())
    else:
        print('Modules: regional_integration.py, posterior_context.py, artifacts.py')
        print((CHECKOUT/'docs/REGIONAL_INTEGRATION_PLAN.md').read_text())
    print('Frontal VMD only; stopped AutoVMD; no model or confirmation qualification')
elif CONFIGURATION.get('schema_version') == 3:
    print('Modules: frontal_campaign.py -> frontal_methods.py -> frontal_replication.py/frontal_native.py')
    print((CHECKOUT/'docs/FRONTAL_PAPER_AUDIT.md').read_text())
    print('Scope: verified frontal only; AutoVMD stopped; posterior unchanged')
else:
    print('Algorithm explanation: docs/SCIENTIFIC_EXPLANATION.md in the pinned checkout')
    print('Modules: signal.py -> reference.py/frontal.py/posterior.py -> experiments.py -> review evidence')
print('Git source:', ACTUAL_SHA)


## Execute reusable modules
HEOG/VEOG are teacher/evaluation information. Clean targets cannot enter inference. Failed numerical or scientific gates stop dependent work.

In [ ]:
command = [str(ENV_PY),'-m','vmd_eog.runner','--stage',RUN_SPEC['stage'],
           '--input','/kaggle/input','--output',str(OUT),'--profile',RUN_SPEC['profile'],
           '--config',RUN_SPEC.get('config_path','configs/campaign.json'),
           '--experiment',str(OUT/'experiment.json')]
process = subprocess.run(command,cwd=CHECKOUT,env=PROCESS_ENV)
if process.returncode:
    raise RuntimeError('Stage failed; inspect saved execution_state.json and kernel logs')
print((OUT/'execution_state.json').read_text())


## Saved configurations and result tables
These are immutable outputs from this run. Readiness, teacher eligibility and final scientific qualification are distinct decisions.

In [ ]:
from IPython.display import display
import pandas as pd
for name in ('coverage_method_means.csv', 'coverage_condition_means.csv', 'coverage_uncertainty_intervals.csv', 'coverage_example_scores.csv'):
    path = OUT/name
    if path.exists():
        print(name, '(first 20 rows; complete table saved)')
        display(pd.read_csv(path).head(20))
if (OUT/'coverage_controlled_gates.json').exists():
    print((OUT/'coverage_controlled_gates.json').read_text())


## Inspect evidence and persist outputs
Native real EEG has no clean-reference SNR. Pass-through failures remain counted. Parameters and physiological frequency bands are distinct.

In [ ]:
from IPython.display import display, Image
for figure in sorted(OUT.glob('*.png')):
    display(Image(filename=str(figure)))
for report in sorted(OUT.glob('*summary.json')):
    print(report.name, report.read_text()[:16000])
print('Persist this completed kernel version and retrieve outputs before launching a dependent stage.')
